# 03 · Async, await and streaming
**⏱ 12 minutes · Concurrent I/O without confusing it with multithreading**

Most AI API calls spend more time **waiting for a server** than doing CPU calculations. You're probably familiar with `CompletableFuture` in Java: start independent work, then combine results.

Python has a related idea in `asyncio`, but with a different execution model. Here we'll *watch* the difference and then practice consuming small chunks of a response.

## 1. Begin with the Java mental model

**Java:**

```java
CompletableFuture<String> a = CompletableFuture.supplyAsync(() -> fetch("a"));
CompletableFuture<String> b = CompletableFuture.supplyAsync(() -> fetch("b"));
String result = a.join() + b.join();
```

That Java example uses worker threads. Python `asyncio` commonly uses **cooperative concurrency on one event-loop thread**. At an `await`, one task can pause while another makes progress. It's great for network I/O but **doesn't magically parallelize CPU-heavy computations**.

Start with a fake network request. `asyncio.sleep` represents waiting, without blocking the event loop.

In [ ]:
import asyncio
import time

async def fake_llm_call(name: str, latency: float = 0.12) -> str:
    print(f"Started {name}")
    await asyncio.sleep(latency)
    print(f"Finished {name}")
    return name.upper()

print("Function defined; no network call has run.")

## 2. Sequential waits: simple, but slower

Calling an `async def` returns a **coroutine object**. We `await` it to run it. Below, each call finishes **before the next starts**.

**Predict:** for three 0.12-second waits, will the total be nearer 0.12 or 0.36 seconds?

In [ ]:
started = time.perf_counter()
sequential = [
    await fake_llm_call("a"),
    await fake_llm_call("b"),
    await fake_llm_call("c"),
]
sequential_seconds = time.perf_counter() - started
print("Sequential:", sequential, round(sequential_seconds, 2), "seconds")
assert sequential == ["A", "B", "C"]

## 3. Concurrent I/O with gather

Now picture three independent AI inputs: account context, a knowledge-base result and permissions. We can *schedule them together* with `asyncio.gather()`.

One interesting detail: **gather returns values in argument order, not completion order.** Watch which call finishes first, and compare that with the resulting list.

> In production, a concurrency limit and backoff policy matter; creating thousands of tasks at once is not free.

In [ ]:
started = time.perf_counter()
concurrent = await asyncio.gather(
    fake_llm_call("a", 0.12),
    fake_llm_call("b", 0.04),
    fake_llm_call("c", 0.08),
)
concurrent_seconds = time.perf_counter() - started
print("Concurrent:", concurrent, round(concurrent_seconds, 2), "seconds")
assert list(concurrent) == ["A", "B", "C"]
assert concurrent_seconds < sequential_seconds

## 4. What about failures and timeouts?

Java APIs may use `orTimeout`, exception handlers and retries. In Python, `asyncio.wait_for()` places a timeout around an awaitable. Here we intentionally trigger a timeout to learn its shape.

**Your turn:** change the timeout from 0.01 to 0.2, then think about how the caller could recover. Restore 0.01 afterward so the exercise's assertion behaves as intended.

In [ ]:
try:
    await asyncio.wait_for(fake_llm_call("slow", 0.1), timeout=0.01)
except asyncio.TimeoutError:
    print("Controlled timeout; decide how the caller should recover.")
else:
    raise AssertionError("The simulated call should time out")

## 5. Generators: the intuition behind streaming

A Java `Stream` is lazy; Python `yield` can also deliver values lazily. A regular function with `return` finishes with **one result**. A generator with `yield` produces values **one by one**.

AI APIs often stream tokens or events so users don't have to wait for the complete answer. A streaming SDK might give you an ordinary generator or an **async iterator**, depending on the library.

**Predict:** is the entire list created before the first loop iteration below?

In [ ]:
def stream_words(text: str):
    for word in text.split():
        yield word

parts = list(stream_words("Streaming makes feedback feel faster"))
for chunk in stream_words("Words arrive gradually"):
    print(chunk, end=" | ")
print()
assert parts == ["Streaming", "makes", "feedback", "feel", "faster"]

### Challenge: an async generator

An **async generator** can wait between pieces, and the consumer uses `async for`. That's much closer to what an asynchronously streaming LLM client might expose.

Try changing the delay and count the chunks. What differs from a coroutine that returns a single string?

In [ ]:
async def stream_async(text: str):
    for word in text.split():
        await asyncio.sleep(0.01)
        yield word

received = []
async for item in stream_async("Three simulated tokens"):
    received.append(item)
    print("Chunk:", item)
assert received == ["Three", "simulated", "tokens"]

## 6. Notebook vs Python module

Notebooks commonly allow **top-level `await`**, as in this lesson. In a normal `.py` module, create an async entrypoint and call it from the standard main guard:

```python
async def main():
    return await fake_llm_call("example")

if __name__ == "__main__":
    asyncio.run(main())
```

**Checkpoint:** would you use `gather()` when the second request *needs the first request's output*? Why not?

**Next:** [04 · Production Python](04_production_python.ipynb). If this still feels abstract, [the async guide](../guides/03_async_and_streaming.md) shows the difference step by step.